In [1]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
Reason for being yanked: Missing dependency on langgraph-prebuilt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 71.2 MB/s

In [1]:
# Import typing helpers for state
from typing import TypedDict, List

# Import timeout support from the standard library
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

# Import LangGraph builder tools
from langgraph.graph import StateGraph, START, END


# Shared notebook for a tiny campus status workflow
class StatusState(TypedDict):
    request: str  # User request text
    api_status: str  # Raw status from the simulated API
    result: str  # Final user-facing message
    trace: List[str]  # Nodes visited
    error: str  # Clear error text when something fails


# Simulated slow API call (stands in for a real HTTP request)
def fake_status_api(request_text: str) -> str:
    import time  # Local import to keep the demo self-contained
    time.sleep(3)  # Pretend the remote API is slow
    return "OPEN"  # Demo status payload


# Run a function with a maximum wait time in seconds
def run_with_timeout(fn, seconds: float, *args):
    with ThreadPoolExecutor(max_workers=1) as pool:  # One worker thread
        future = pool.submit(fn, *args)  # Start the work
        return future.result(timeout=seconds)  # Wait only up to the limit


# Node 1: prepare the request
def understand_request(state: StatusState) -> dict:
    cleaned = state["request"].strip()  # Clean spaces
    return {
        "request": cleaned,  # Store cleaned request
        "trace": state["trace"] + ["understand_request"],  # Log node
        "error": "",  # Clear old errors at the start
    }


# Node 2: API-backed step with a SHORT timeout (will fail in this demo)
def fetch_status(state: StatusState) -> dict:
    try:  # Attempt the timed API call
        status = run_with_timeout(fake_status_api, 1.0, state["request"])  # 1 second limit
        return {
            "api_status": status,  # Save API payload
            "trace": state["trace"] + ["fetch_status"],  # Log success path
            "error": "",  # No error
        }
    except FuturesTimeout:  # Timer rang before the API finished
        return {
            "api_status": "",  # No useful payload
            "trace": state["trace"] + ["fetch_status"],  # Still record the attempt
            "error": "Status service took too long. Please try again in a minute.",  # User-facing timeout message
        }


# Node 3: turn state into a final message
def write_result(state: StatusState) -> dict:
    if state["error"]:  # Prefer clear error if present
        message = state["error"]  # Show the timeout (or other) error
    elif state["api_status"]:  # Happy path
        message = "Campus desk status: " + state["api_status"]  # Show status
    else:  # Unexpected empty path
        message = "Could not read campus desk status."  # Fallback text
    return {
        "result": message,  # Final message
        "trace": state["trace"] + ["write_result"],  # Log node
    }


# Build the graph
builder = StateGraph(StatusState)  # Shell
builder.add_node("understand_request", understand_request)  # Node 1
builder.add_node("fetch_status", fetch_status)  # Timed API node
builder.add_node("write_result", write_result)  # Final message node
builder.add_edge(START, "understand_request")  # Wire
builder.add_edge("understand_request", "fetch_status")  # Wire
builder.add_edge("fetch_status", "write_result")  # Wire
builder.add_edge("write_result", END)  # Wire

# Compile and run
graph = builder.compile()
final_state = graph.invoke(
    {
        "request": "Is the scholarship desk open?",  # User goal
        "api_status": "",  # Empty before API
        "result": "",  # Empty before final node
        "trace": [],  # Fresh trace
        "error": "",  # No error yet
    }
)

print("TRACE:", final_state["trace"])  # Show path
print("ERROR:", final_state["error"])  # Expect timeout message
print("RESULT:", final_state["result"])  # User-facing text

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


TRACE: ['understand_request', 'fetch_status', 'write_result']
ERROR: Status service took too long. Please try again in a minute.
RESULT: Status service took too long. Please try again in a minute.


In [ ]:
final_state

{'request': 'Is the scholarship desk open?',
 'api_status': '',
 'result': 'Status service took too long. Please try again in a minute.',
 'trace': ['understand_request', 'fetch_status', 'write_result'],
 'error': 'Status service took too long. Please try again in a minute.'}

In [2]:
# Import timeout tools
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

# Import the compiled graph object from your module OR rebuild a tiny graph below
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END


# Tiny state for the global-timeout demo
class TinyState(TypedDict):
    value: int  # Simple counter value
    trace: List[str]  # Trace list


# A deliberately slow node
def slow_step(state: TinyState) -> dict:
    import time  # Local import for the demo
    time.sleep(2)  # Slow work
    return {
        "value": state["value"] + 1,  # Update value
        "trace": state["trace"] + ["slow_step"],  # Log node
    }


# Build a one-node graph
builder = StateGraph(TinyState)  # Shell
builder.add_node("slow_step", slow_step)  # Only node
builder.add_edge(START, "slow_step")  # Start
builder.add_edge("slow_step", END)  # End
graph = builder.compile()  # Compile


# Run any invoke with a maximum total wait
def invoke_with_global_timeout(graph_obj, state: dict, seconds: float) -> dict:
    with ThreadPoolExecutor(max_workers=1) as pool:  # Worker pool
        future = pool.submit(graph_obj.invoke, state)  # Start full run
        return future.result(timeout=seconds)  # Enforce global limit


# Attempt with a timeout that is too small
try:  # Catch global timeout for a user-facing message
    result = invoke_with_global_timeout(
        graph,  # Graph to run
        {"value": 0, "trace": []},  # Start state
        0.5,  # Global limit too short for sleep(2)
    )
    print("UNEXPECTED SUCCESS:", result)  # Should not happen in this demo
except FuturesTimeout:  # Whole run exceeded the limit
    user_message = (
        "Sorry — this request timed out before completion. "
        "Please try again in a little while."
    )  # Calm user-facing error
    print("USER MESSAGE:", user_message)  # Show what you would display

USER MESSAGE: Sorry — this request timed out before completion. Please try again in a little while.
